In [37]:
import pandas as pd
from sqlalchemy import text
from sqlalchemy import create_engine,text
from urllib.parse import quote_plus
import time

In [22]:


connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost,1433;"
    "DATABASE=DataWarehouse;"
    "Trusted_Connection=yes;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(connection_string)
)



In [27]:
query = text("""
    SELECT TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'raw'
      AND TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_NAME
""")

with engine.connect() as conn:
    tables_df = pd.read_sql(query, conn)

tables_df

,TABLE_NAME
0,crm_cust_info
1,crm_prd_info
2,crm_sales_details
3,erp_cust_az12
4,erp_loc_a101
5,erp_px_cat_g1v2


In [29]:
name_mapping={'cust_info':'crm',
'prd_info':'crm',
'sales_details':'crm',
'cust_az12':'erp',
'loc_a101':'erp',
'px_cat_g1v2':'erp'}

In [31]:
names=list(name_mapping.keys())
names

['cust_info',
 'prd_info',
 'sales_details',
 'cust_az12',
 'loc_a101',
 'px_cat_g1v2']

In [39]:
batch_start_time = time.perf_counter()

print('='*50)
print('Loading Raw Layer')
print('='*50)

for name in names:
    start_time = time.perf_counter()

    print("-" * 50)

    source=name_mapping[name]
    if source =='erp':
        name=name.upper()
    try:
        file=pd.read_csv(rf"C:\Users\alice.romagnoni\Desktop\sql-data-warehouse-project\datasets\source_{source}\{name}.csv")

        table_name=source+'_'+name.lower()

        with engine.begin() as conn:
            print('Truncating Table: '+table_name)
            conn.execute(text(f"TRUNCATE TABLE raw.{table_name}"))

            print('Loading Table: '+table_name)
            file.to_sql(name=table_name,schema='raw',con=conn,if_exists='append',index=False)


        difference_time = time.perf_counter() - start_time

        print(f"Rows loaded: {len(file):,}")
        print(f"Load Duration: {difference_time:.2f} seconds")
        print("-" * 50)

    except Exception as exc:
        print(f"\nERROR while processing: {table_name}")
        print(f"Error type: {type(exc).__name__}")
        print(f"Error message: {exc}")

total_difference_time = time.perf_counter() - batch_start_time

print('Loading Raw Layer Completed')
print(f"Total Load Duration: : {total_difference_time:.2f} seconds")

   

Loading Raw Layer
--------------------------------------------------
Truncating Table: crm_cust_info
Loading Table: crm_cust_info
Rows loaded: 18,494
Load Duration: 2.72 seconds
--------------------------------------------------
--------------------------------------------------
Truncating Table: crm_prd_info
Loading Table: crm_prd_info
Rows loaded: 397
Load Duration: 0.33 seconds
--------------------------------------------------
--------------------------------------------------
Truncating Table: crm_sales_details
Loading Table: crm_sales_details
Rows loaded: 60,398
Load Duration: 4.75 seconds
--------------------------------------------------
--------------------------------------------------
Truncating Table: erp_cust_az12
Loading Table: erp_cust_az12
Rows loaded: 18,484
Load Duration: 0.98 seconds
--------------------------------------------------
--------------------------------------------------
Truncating Table: erp_loc_a101
Loading Table: erp_loc_a101
Rows loaded: 18,484
Load 